In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/train.txt' ,sep = ';', header = None , names = ('text' , 'emotions'))
print(df.head())

print("checking the null values ")
print(df.isnull().sum())


                                                text emotions
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i am ever feeling nostalgic about the fireplac...     love
4                               i am feeling grouchy    anger
checking the null values 
text        0
emotions    0
dtype: int64


In [ ]:
#text clening
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['emotions'] = le.fit_transform(df['emotions'])
print(df.head())

                                                text  emotions
0                            i didnt feel humiliated         4
1  i can go from feeling so hopeless to so damned...         4
2   im grabbing a minute to post i feel greedy wrong         0
3  i am ever feeling nostalgic about the fireplac...         3
4                               i am feeling grouchy         0


In [ ]:
#lower casing
import string

df['text'] = df['text'].str.lower()
print(df.head())

                                                text  emotions
0                            i didnt feel humiliated         4
1  i can go from feeling so hopeless to so damned...         4
2   im grabbing a minute to post i feel greedy wrong         0
3  i am ever feeling nostalgic about the fireplac...         3
4                               i am feeling grouchy         0


In [ ]:
#remove punctuation
df['text'] = df['text'].str.translate(str.maketrans('' , '' , string.punctuation))
print(df.head())

                                                text  emotions
0                            i didnt feel humiliated         4
1  i can go from feeling so hopeless to so damned...         4
2   im grabbing a minute to post i feel greedy wrong         0
3  i am ever feeling nostalgic about the fireplac...         3
4                               i am feeling grouchy         0


In [ ]:
#remove number
df['text'] = df['text'].str.replace(r'\d+' , '', regex=True)
print(df.head())

def remove_number(text):
  result = ''.join([i for i in text if not i.isdigit()])
  return result


df['text'] = df['text'].apply(remove_number)
print(df.head())

                                                text  emotions
0                            i didnt feel humiliated         4
1  i can go from feeling so hopeless to so damned...         4
2   im grabbing a minute to post i feel greedy wrong         0
3  i am ever feeling nostalgic about the fireplac...         3
4                               i am feeling grouchy         0
                                                text  emotions
0                            i didnt feel humiliated         4
1  i can go from feeling so hopeless to so damned...         4
2   im grabbing a minute to post i feel greedy wrong         0
3  i am ever feeling nostalgic about the fireplac...         3
4                               i am feeling grouchy         0


In [ ]:
#remove emojis
import re

def remove_emoji(text):
  new = ""
  for i in text:
    if i.isascii():
      new += i
  return new

df['text'] = df['text'].apply(remove_emoji)
print(df.head())


                                                text  emotions
0                            i didnt feel humiliated         4
1  i can go from feeling so hopeless to so damned...         4
2   im grabbing a minute to post i feel greedy wrong         0
3  i am ever feeling nostalgic about the fireplac...         3
4                               i am feeling grouchy         0


In [ ]:
#remove stopwords
import nltk
from nltk.corpus import stopwords # Import stopwords

nltk.download('stopwords')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))

def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if i not in stop_words:
       cleaned.append(i)
  return ' '.join(cleaned)

df['text'] = df['text'].apply(remove) # Corrected dff to df and moved out of function
print(df.head())

                                                text  emotions
0                              didnt feel humiliated         4
1  go feeling hopeless damned hopeful around some...         4
2          im grabbing minute post feel greedy wrong         0
3  ever feeling nostalgic fireplace know still pr...         3
4                                    feeling grouchy         0


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [ ]:
df.shape

(16000, 2)

In [ ]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['emotions']

x_train , x_test , y_train , y_test = train_test_split(X , y , test_size = 20 , random_state = 42)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer , CountVectorizer

vectorizer = TfidfVectorizer()
x_train_tf = vectorizer.fit_transform(x_train)
x_test_tf = vectorizer.transform(x_test)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Re-run train_test_split to ensure x_train and x_test contain the original text data
X = df['text']
y = df['emotions']
x_train , x_test , y_train , y_test = train_test_split(X , y , test_size = 20 , random_state = 42)

count = CountVectorizer()
x_train_count = count.fit_transform(x_train)
x_test_count = count.transform(x_test)

idf = TfidfVectorizer()
x_train_idf = idf.fit_transform(x_train)
x_test_idf = idf.transform(x_test)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb = MultinomialNB()
nb.fit(x_train_tf , y_train)
y_pred = nb.predict(x_test_tf)

print(accuracy_score(y_test , y_pred))

0.65


In [ ]:
nb_2 = MultinomialNB()

nb_2.fit(x_train_count , y_train)
y_pred = nb_2.predict(x_test_count)

print(accuracy_score(y_test , y_pred))

0.8


In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(x_train_tf , y_train)
y_pred = lr.predict(x_test_tf)

print(accuracy_score(y_test , y_pred))

0.85


In [ ]:
lr_2 = LogisticRegression(max_iter=1000)

lr_2.fit(x_train_count , y_train)
y_pred = lr_2.predict(x_test_count)

print(accuracy_score(y_test , y_pred))

0.95


In [ ]:
emotion_mapping = {
    0 : 'anger',
    1 : 'fear',
    2 : 'joy',
    3 : 'love',
    4 : 'sadness',
    5 : 'surprise'
}

# Make predictions using the trained Logistic Regression model (lr) on TF-IDF features
# Assuming lr is the model trained on x_train_tf and is available from cell e3x2LYd5N_ty
# If a different model is desired (e.g., nb), use that model's predict method.
predicted_labels = lr.predict(x_test_tf)

# Map the numerical predictions to emotion names using the emotion_mapping dictionary
new_prediction_names = [emotion_mapping[label] for label in predicted_labels]

print(new_prediction_names)

['sadness', 'joy', 'sadness', 'joy', 'sadness', 'sadness', 'fear', 'sadness', 'joy', 'fear', 'sadness', 'anger', 'joy', 'sadness', 'sadness', 'fear', 'anger', 'joy', 'love', 'sadness']
['sadness', 'joy', 'sadness', 'joy', 'sadness', 'sadness', 'fear', 'sadness', 'joy', 'fear', 'sadness', 'anger', 'joy', 'sadness', 'sadness', 'fear', 'anger', 'joy', 'love', 'sadness']


### Predict Emotion for New User Input

In [ ]:
# Take user input
user_input = input("Enter your text here to predict its emotion: ")

# Preprocess the user input (replicating earlier steps)
processed_input = user_input.lower() # Lowercasing
processed_input = processed_input.translate(str.maketrans('' , '' , string.punctuation)) # Remove punctuation
processed_input = re.sub(r'\d+' , '', processed_input) # Remove numbers (using regex as in cell 862XgwP3bd_P)
processed_input = remove_number(processed_input) # Redundant due to regex, but keeping for consistency with notebook
processed_input = remove_emoji(processed_input) # Remove emojis
processed_input = remove_stopwords_func(processed_input) # Remove stopwords

# Vectorize the preprocessed input using the same TF-IDF vectorizer ('vectorizer' from cell Dvm-zaZbMhM4)
input_vectorized = vectorizer.transform([processed_input])

# Predict the emotion using the trained Logistic Regression model ('lr' from cell e3x2LYd5N_ty)
predicted_label_numeric = lr.predict(input_vectorized)[0]

# Map the numerical prediction to an emotion name using the 'emotion_mapping' dictionary
predicted_emotion_name = emotion_mapping[predicted_label_numeric]

print(f"The predicted emotion for your text is: {predicted_emotion_name}")

Enter your text here to predict its emotion: i am happy
The predicted emotion for your text is: joy
